# LightGBM with Demand Typology Features

## Overview

This notebook extends the baseline anomaly detection pipeline from `LightGBM_Final_Methodology_Test.ipynb` by incorporating the product demand typology developed in `Product_Clustering_Method.ipynb`. The hypothesis is that encoding each product's seasonal archetype — monthly pattern, holiday concentration, weekly rhythm, and intraday peak — as explicit model features will improve the quality of residual-based anomaly detection.

The baseline model (RMSE Strong, z = −4.5) achieved 90.8% recall on 65 labelled anomalies with a mean historical z-score of 4.15. This notebook trains an identical configuration augmented with four typology features and compares the results using the same evaluation framework. Only one configuration is tested, keeping the experiment focused and the comparison clean.

In [11]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import gc
import warnings
warnings.filterwarnings('ignore')

DATA_PATH     = r'C:\Users\User\Desktop\диплом\data_v1.csv'
AUG_FULL_PATH = r'C:\Users\User\Desktop\диплом\df_aug_final.csv'
AUG_ZERO_PATH = r'C:\Users\User\Desktop\диплом\df_aug_final_check_rows.csv'

# аномалии базовой модели (сохранены в LightGBM_Final_Methodology_Test.ipynb)
BASELINE_PATH = r'C:\Users\User\Desktop\диплом\LightGBM_final_test_anomalies_z45\3_rmse_strong_z45.csv'

RANDOM_STATE = 42
print("LightGBM version:", lgb.__version__)

LightGBM version: 4.6.0


### Data Loading

Identical cold-start filtering as in the baseline notebook. The demand typology table is reconstructed from the full dataset using the same logic as `Product_Clustering_Method.ipynb` — no external file dependency is introduced.

In [2]:
df = pd.read_csv(DATA_PATH)
df['date']    = pd.to_datetime(df['date'])
df['product'] = df['product'].astype(str)

first_pos = df[df['sales'] > 0].groupby('product')['date'].min()
cold      = first_pos[first_pos >= '2025-08-01'].index
df        = df[~df['product'].isin(cold)].copy()
df        = df[df['date'] <= '2025-08-31'].copy()

df_aug      = pd.read_csv(AUG_FULL_PATH)
df_aug_zero = pd.read_csv(AUG_ZERO_PATH)
df_aug['date']      = pd.to_datetime(df_aug['date'])
df_aug_zero['date'] = pd.to_datetime(df_aug_zero['date'])
df_aug['product']      = df_aug['product'].astype(str)
df_aug_zero['product'] = df_aug_zero['product'].astype(str)

print(f"Dataset: {df.shape[0]:,} rows | {df['product'].nunique():,} products")

Dataset: 41,469,926 rows | 7,775 products


### Demand Typology Construction

Four typology labels are computed from active-hour observations (sales > 0) using the same methodology as `Product_Clustering_Method.ipynb`. Labels are encoded as integers for LightGBM compatibility.

In [6]:
df['hour']      = df['date'].dt.hour
df['dayofweek'] = df['date'].dt.dayofweek
df['month']     = df['date'].dt.month

df_active = df[df['sales'] > 0].copy()

# доли по дням недели
dow_active = (df_active.groupby(['product', 'dayofweek'])['sales']
              .count().unstack(fill_value=0))
dow_active = dow_active.div(dow_active.sum(axis=1), axis=0)
dow_active.columns = [f'dow_share_{c}' for c in dow_active.columns]
dow_active = dow_active.reset_index()

# доли по месяцам
month_active = (df_active.groupby(['product', 'month'])['sales']
                .count().unstack(fill_value=0))
month_active = month_active.div(month_active.sum(axis=1), axis=0)
month_active.columns = [f'month_share_{c}' for c in month_active.columns]
month_active = month_active.reset_index()

# доли по часам
hour_active = (df_active.groupby(['product', 'hour'])['sales']
               .count().unstack(fill_value=0))
hour_active = hour_active.div(hour_active.sum(axis=1), axis=0)
hour_active.columns = [f'hour_share_{c}' for c in hour_active.columns]
hour_active = hour_active.reset_index()

fa = dow_active.merge(month_active, on='product', how='left') \
               .merge(hour_active,  on='product', how='left') \
               .fillna(0)

def safe_sum(df, cols):
    existing = [c for c in cols if c in df.columns]
    return df[existing].sum(axis=1) if existing else pd.Series(0, index=df.index)

fa['share_winter'] = safe_sum(fa, ['month_share_1','month_share_2','month_share_12'])
fa['share_spring'] = safe_sum(fa, ['month_share_3','month_share_4','month_share_5'])
fa['share_summer'] = safe_sum(fa, ['month_share_6','month_share_7','month_share_8'])
fa['share_autumn'] = safe_sum(fa, ['month_share_9','month_share_10','month_share_11'])

def monthly_type(row, thr=0.45):
    s = {k: row[f'share_{k}'] for k in ['winter','spring','summer','autumn']}
    best = max(s, key=s.get)
    return best if s[best] >= thr else 'none'

fa['monthly_type'] = fa.apply(monthly_type, axis=1)

fa['share_holiday'] = safe_sum(fa, ['month_share_2','month_share_3',
                                     'month_share_5','month_share_12'])
summer_cols = [c for c in ['month_share_6','month_share_7','month_share_8',
                             'month_share_9','month_share_10'] if c in fa.columns]
fa['is_holiday_driven'] = (
    (fa['share_holiday'] > 0.55) &
    (fa[summer_cols].max(axis=1) < 0.12 if summer_cols else True)
).astype(int)

fa['share_fri_sat'] = fa[['dow_share_4','dow_share_5']].sum(axis=1)
fa['share_weekend']  = fa[['dow_share_5','dow_share_6']].sum(axis=1)
fa['share_weekdays'] = fa[['dow_share_0','dow_share_1','dow_share_2',
                             'dow_share_3','dow_share_4']].sum(axis=1)

def weekly_type(row, thr=0.35):
    if row['share_fri_sat'] >= thr:   return 'fri_sat_peak'
    if row['share_weekend']  >= thr:  return 'weekend_peak'
    if row['share_weekdays'] >= 0.75: return 'weekday_focused'
    return 'uniform'

fa['weekly_type'] = fa.apply(weekly_type, axis=1)

hour_cols_h = [c for c in fa.columns if c.startswith('hour_share_')]
fa['share_morning']   = fa[[c for c in hour_cols_h if int(c.split('_')[-1]) in range(6,12)]].sum(axis=1)
fa['share_afternoon'] = fa[[c for c in hour_cols_h if int(c.split('_')[-1]) in range(12,17)]].sum(axis=1)
fa['share_evening']   = fa[[c for c in hour_cols_h if int(c.split('_')[-1]) in range(17,23)]].sum(axis=1)

def hourly_type(row, thr=0.40):
    s = {'morning': row['share_morning'],
         'afternoon': row['share_afternoon'],
         'evening': row['share_evening']}
    best = max(s, key=s.get)
    return best if s[best] >= thr else 'uniform'

fa['hourly_type'] = fa.apply(hourly_type, axis=1)

# кодируем в числа для LightGBM
monthly_map = {'none':0,'winter':1,'spring':2,'summer':3,'autumn':4}
weekly_map  = {'uniform':0,'weekday_focused':1,'fri_sat_peak':2,'weekend_peak':3}
hourly_map  = {'uniform':0,'morning':1,'afternoon':2,'evening':3}

fa['monthly_type_enc'] = fa['monthly_type'].map(monthly_map).astype('int8')
fa['weekly_type_enc']  = fa['weekly_type'].map(weekly_map).astype('int8')
fa['hourly_type_enc']  = fa['hourly_type'].map(hourly_map).astype('int8')

typology = fa[['product','monthly_type_enc','weekly_type_enc',
               'hourly_type_enc','is_holiday_driven']].copy()

print("Typology features computed:")
print(typology.dtypes)
print(f"\nMonthly: {fa['monthly_type'].value_counts().to_dict()}")
print(f"Weekly:  {fa['weekly_type'].value_counts().to_dict()}")
print(f"Hourly:  {fa['hourly_type'].value_counts().to_dict()}")

del df_active, dow_active, month_active, hour_active, fa
gc.collect()

Typology features computed:
product                str
monthly_type_enc      int8
weekly_type_enc       int8
hourly_type_enc       int8
is_holiday_driven    int64
dtype: object

Monthly: {'summer': 4405, 'spring': 1399, 'none': 1134, 'winter': 837}
Weekly:  {'uniform': 4037, 'weekday_focused': 2295, 'fri_sat_peak': 875, 'weekend_peak': 568}
Hourly:  {'evening': 2993, 'uniform': 2449, 'afternoon': 2047, 'morning': 286}


1123

### Product Profile and Full Feature Engineering

Identical to the baseline notebook. Typology features are merged into the product profile table and propagated to both train and test sets alongside all existing features.

In [7]:
df_til_sep = df[df['date'].dt.month < 8].copy()

products    = df_til_sep[['product','category']].drop_duplicates()
first_sales = (df_til_sep[df_til_sep['sales'] > 0]
               .groupby('product')['date'].min().reset_index())
first_sales.columns = ['product','first_sale_date']
products = products.merge(first_sales, on='product', how='left')

df_act = df_til_sep.merge(first_sales, on='product', how='left')
df_act = df_act[df_act['date'] >= df_act['first_sale_date']]

all_features = df_act.groupby('product').agg(
    hist_mean     = ('sales','mean'),
    hist_std      = ('sales','std'),
    hist_cv       = ('sales', lambda x: x.std()/(x.mean()+1e-6)),
    hist_zero     = ('sales', lambda x: (x==0).mean()),
    hist_adi      = ('sales', lambda x: (x==0).sum()/max((x>0).sum(),1)),
    hist_max      = ('sales','max'),
    hist_p99      = ('sales', lambda x: np.percentile(x,99)),
    hist_peak     = ('sales', lambda x: x.max()/(x.mean()+1e-6)),
    hist_outlier  = ('sales', lambda x: (x > np.percentile(x,99)).mean()),
    hist_pos_rate = ('sales', lambda x: (x>0).mean()),
    accel         = ('sales', lambda x: x.iloc[-72:].mean()/(x.iloc[:72].mean()+1e-6) if len(x)>=144 else 1.0),
    recent_vs_avg = ('sales', lambda x: x.iloc[-72:].mean()/(x.mean()+1e-6)),
    vol_3d        = ('sales', lambda x: x.iloc[-72:].std()/(x.iloc[-72:].mean()+1e-6) if len(x)>=72 else 0.0),
).reset_index()

products = products.merge(all_features, on='product', how='left')

# +++ типология +++
products = products.merge(typology, on='product', how='left')
for col in ['monthly_type_enc','weekly_type_enc','hourly_type_enc','is_holiday_driven']:
    products[col] = products[col].fillna(0).astype('int8')

del df_act, all_features, first_sales
gc.collect()

profile_cols = ['hist_mean','hist_std','hist_cv','hist_zero','hist_adi','hist_max',
                'hist_p99','hist_peak','hist_outlier','hist_pos_rate',
                'accel','recent_vs_avg','vol_3d']
for col in profile_cols:
    med = products.groupby('category')[col].median()
    products.loc[products[col].isna(), col] = products.loc[products[col].isna(),'category'].map(med)
products[profile_cols] = products[profile_cols].replace([np.inf,-np.inf],np.nan).fillna(0).astype('float32')

# временные фичи
df_til_sep = df[df['date'] <= '2025-08-31'].copy()
df_til_sep = (df_til_sep.merge(products, on=['product','category'], how='left')
              .sort_values(['product','date']).reset_index(drop=True))

df_til_sep['hour']       = df_til_sep['date'].dt.hour
df_til_sep['dayofweek']  = df_til_sep['date'].dt.dayofweek
df_til_sep['is_weekend'] = (df_til_sep['dayofweek'] >= 5).astype(int)
df_til_sep['week']       = df_til_sep['date'].dt.isocalendar().week

for lag in [1, 24, 168]:
    df_til_sep[f'sales_lag_{lag}h'] = df_til_sep.groupby('product')['sales'].shift(lag)

df_til_sep['stocks_lag_1h'] = df_til_sep.groupby('product')['stocks'].shift(1).fillna(0)

for w in [6, 24, 168]:
    mp = max(1, w//2)
    df_til_sep[f'rolling_mean_{w}h'] = df_til_sep.groupby('product')['sales'].transform(
        lambda x: x.rolling(w, min_periods=mp).mean())
    df_til_sep[f'rolling_std_{w}h']  = df_til_sep.groupby('product')['sales'].transform(
        lambda x: x.rolling(w, min_periods=mp).std())

df_til_sep['z_score_24h']      = (df_til_sep['sales'] - df_til_sep['rolling_mean_24h']) / (df_til_sep['rolling_std_24h'] + 1e-6)
df_til_sep['pct_change_1h']    = df_til_sep.groupby('product')['sales'].pct_change(1)
df_til_sep['pct_change_24h']   = df_til_sep.groupby('product')['sales'].pct_change(24)
df_til_sep['sales_vs_mean_24h']= df_til_sep['sales'] / (df_til_sep['rolling_mean_24h'] + 1e-6)

num_cols = df_til_sep.select_dtypes(include=['float32','float64','int32','int64']).columns
df_til_sep[num_cols] = df_til_sep[num_cols].replace([np.inf,-np.inf],np.nan).fillna(0)

### Train / Test Split

In [8]:
df_train = df_til_sep[df_til_sep['date'].dt.month < 8].copy()
df_test  = df_aug.copy()

cols_to_add = [c for c in df_til_sep.columns
               if c not in ['product','date','category','stocks','sales','price']]
df_test = df_test.merge(df_til_sep[['product','date'] + cols_to_add],
                        on=['product','date'], how='left')
df_test = df_test.sort_values(['product','date'])
df_test['stocks_lag_1h'] = df_test.groupby('product')['stocks'].shift(1).fillna(0)
df_test['hour']       = df_test['date'].dt.hour
df_test['dayofweek']  = df_test['date'].dt.dayofweek
df_test['is_weekend'] = (df_test['dayofweek'] >= 5).astype(int)
df_test['week']       = df_test['date'].dt.isocalendar().week

exclude  = ['product','date','category','first_sale_date','sales']
features = [c for c in df_til_sep.columns if c not in exclude]

X_train, y_train = df_train[features], df_train['sales']
X_test,  y_test  = df_test[features],  df_test['sales']

typology_features = ['monthly_type_enc','weekly_type_enc',
                     'hourly_type_enc','is_holiday_driven']
print(f"Train: {len(X_train):,} | Test: {len(X_test):,} | Features: {len(features)}")
print(f"Typology features included: {[f for f in typology_features if f in features]}")

del df_til_sep, products
gc.collect()

Train: 36,388,440 | Test: 5,287,704 | Features: 38
Typology features included: ['monthly_type_enc', 'weekly_type_enc', 'hourly_type_enc', 'is_holiday_driven']


0

### Model Training: RMSE Strong with Typology Features

The RMSE Strong configuration is selected as the baseline to extend, as it achieved the highest mean |z_hist| (4.15) in the original benchmark. Parameters are identical to the baseline; the only difference is the four additional typology features in the training matrix.

In [9]:
params = {
    'objective':        'regression',
    'metric':           'mae',
    'learning_rate':    0.05,
    'n_estimators':     1200,
    'num_leaves':       48,
    'max_depth':        7,
    'min_child_samples':150,
    'reg_alpha':        0.3,
    'reg_lambda':       0.8,
    'min_split_gain':   0.05,
    'subsample':        0.75,
    'colsample_bytree': 0.65,
    'path_smooth':      8,
    'max_bin':          255,
    'verbose':          -1,
    'random_state':     RANDOM_STATE,
    'n_jobs':           -1,
}

val_mask         = (df_train['date'] >= '2025-07-25') & (df_train['date'] < '2025-08-01')
X_val            = df_train.loc[val_mask, features]
y_val            = df_train.loc[val_mask, 'sales']
sample_weight    = np.where(y_train > 0, 3, 1)
sample_weight_val= np.where(y_val   > 0, 3, 1)

model = lgb.LGBMRegressor(**params)
model.fit(
    X_train, y_train,
    sample_weight=sample_weight,
    eval_set=[(X_val, y_val)],
    eval_sample_weight=[sample_weight_val],
    callbacks=[
        lgb.early_stopping(stopping_rounds=10, verbose=False),
        lgb.log_evaluation(period=100),
    ]
)
print(f"Best iteration: {model.best_iteration_}")

[100]	valid_0's l1: 0.0108136
[200]	valid_0's l1: 0.00879926
[300]	valid_0's l1: 0.00806866
[400]	valid_0's l1: 0.00765886
[500]	valid_0's l1: 0.00735086
[600]	valid_0's l1: 0.00709612
[700]	valid_0's l1: 0.00691366
[800]	valid_0's l1: 0.00675399
[900]	valid_0's l1: 0.00661689
[1000]	valid_0's l1: 0.00649769
[1100]	valid_0's l1: 0.00635039
[1200]	valid_0's l1: 0.00622386
Best iteration: 1200


### Anomaly Detection and Comparison

Anomalies are detected at z = −4.5 using the same mandatory domain filter (stock > 0 in current and preceding hour, business hours 07:00–22:00). Results are compared against the baseline RMSE Strong model using recall on 65 labelled events and the full statistical diagnostic framework.

In [12]:
df_test_local = df_test.copy()
df_test_local['predicted_sales']  = model.predict(X_test)
df_test_local['residual']         = df_test_local['sales'] - df_test_local['predicted_sales']
df_test_local['residual_zscore']  = (
    (df_test_local['residual'] - df_test_local['residual'].mean()) /
     df_test_local['residual'].std()
)

df_test_local['mandatory_filter'] = (
    (df_test_local['stocks_lag_1h'] > 0) &
    (df_test_local['stocks']        > 0) &
    (df_test_local['hour']          >= 7) &
    (df_test_local['hour']          <= 22)
)

Z = -4.5
df_test_local['is_anomaly'] = (
    df_test_local['mandatory_filter'] &
    (df_test_local['residual_zscore'] < Z)
)
anomalies_new = df_test_local[df_test_local['is_anomaly']][
    ['date','product','category','sales','stocks','price']
].copy()

matches_new = len(anomalies_new.merge(
    df_aug_zero[['product','date']], on=['product','date'], how='inner'))

print(f"Typology model | z < {Z}: {len(anomalies_new):,} anomalies | "
      f"{anomalies_new['product'].nunique()} products | "
      f"{matches_new}/65 ({round(matches_new/65*100,1)}% recall)")

# загружаем базовую модель для сравнения
baseline = pd.read_csv(BASELINE_PATH)
baseline['date']    = pd.to_datetime(baseline['date'])
baseline['product'] = baseline['product'].astype(str)
matches_base = len(baseline.merge(
    df_aug_zero[['product','date']], on=['product','date'], how='inner'))

print(f"Baseline model | z < {Z}: {len(baseline):,} anomalies | "
      f"{baseline['product'].nunique()} products | "
      f"{matches_base}/65 ({round(matches_base/65*100,1)}% recall)")

Typology model | z < -4.5: 478 anomalies | 65 products | 60/65 (92.3% recall)
Baseline model | z < -4.5: 450 anomalies | 51 products | 59/65 (90.8% recall)


### Statistical Quality Evaluation

In [13]:
def evaluate_anomalies(anomaly_df, full_df, test_start='2025-08-01'):
    pre_test  = full_df[full_df['date'] < test_start].copy()
    first_pos = (pre_test[pre_test['sales'] > 0]
                 .groupby('product')['date'].min()
                 .reset_index(name='first_positive_date'))
    pre_test  = pre_test.merge(first_pos, on='product', how='left')
    pre_test  = pre_test[pre_test['date'] >= pre_test['first_positive_date']].copy()

    stats_full = pre_test.groupby('product').agg(
        mean_sales_hist   = ('sales','mean'),
        std_sales_hist    = ('sales','std'),
    ).reset_index()
    stats_july = (pre_test[pre_test['date'].dt.month == 7]
                  .groupby('product').agg(
                      mean_sales_july = ('sales','mean'),
                      std_sales_july  = ('sales','std'),
                  ).reset_index())

    anom = anomaly_df.copy()
    anom['date']    = pd.to_datetime(anom['date'])
    anom['product'] = anom['product'].astype(str)
    anom['hour']    = anom['date'].dt.hour
    anom['weekday'] = anom['date'].dt.dayofweek
    anom = anom.merge(stats_full, on='product', how='left')
    anom = anom.merge(stats_july, on='product', how='left')

    for col in ['std_sales_hist','std_sales_july','mean_sales_hist','mean_sales_july']:
        anom[col] = anom[col].replace(0, np.nan).fillna(1)

    anom['z_hist']        = (anom['sales'] - anom['mean_sales_hist']) / anom['std_sales_hist']
    anom['z_july']        = (anom['sales'] - anom['mean_sales_july']) / anom['std_sales_july']
    anom['dev_pct_hist']  = (anom['sales'] - anom['mean_sales_hist']) / anom['mean_sales_hist'] * 100

    stock_prev    = full_df[['date','product','stocks']].rename(
        columns={'stocks':'stocks_prev','date':'prev_date'})
    anom['prev_date'] = anom['date'] - pd.Timedelta(hours=1)
    anom = anom.merge(stock_prev, on=['prev_date','product'], how='left')

    drop_mask      = anom['sales'] < anom['mean_sales_hist']
    zero_mask      = anom['sales'] == 0
    deep_drop_mask = anom['dev_pct_hist'] < -50
    stock_ok_mask  = (anom['stocks_prev'] > 0) & (anom['stocks'] > 0)
    n = len(anom)

    return {
        'Anomalies found':              n,
        'Unique products':              anom['product'].nunique(),
        'Unique days':                  anom['date'].dt.date.nunique(),
        'Anomalies/day (avg)':          round(n / max(anom['date'].dt.date.nunique(),1), 1),
        'Mean deviation from norm (%)': round(anom.loc[drop_mask,'dev_pct_hist'].mean(), 1),
        'Mean |z_hist|':                round(anom['z_hist'].abs().mean(), 3),
        'Mean |z_july|':                round(anom['z_july'].abs().mean(), 3),
        '% |z_hist| > 2':              round((anom['z_hist'].abs() > 2).mean() * 100, 1),
        '% |z_hist| > 3':              round((anom['z_hist'].abs() > 3).mean() * 100, 1),
        '% zero sales':                 round(zero_mask.mean() * 100, 1),
        '% drops > 50% below norm':     round(deep_drop_mask.mean() * 100, 1),
        '% anomalies with stock':       round(stock_ok_mask.mean() * 100, 1),
        'Zero stock at anomaly time':   int((anom['stocks'] == 0).sum()),
        '% working hours (7–22)':       round(((anom['hour'] >= 7) & (anom['hour'] <= 22)).mean() * 100, 1),
        '% weekdays':                   round((anom['weekday'] < 5).mean() * 100, 1),
    }

full_df = pd.read_csv(DATA_PATH)
full_df['date']    = pd.to_datetime(full_df['date'])
full_df['product'] = full_df['product'].astype(str)

m_base = evaluate_anomalies(baseline,      full_df)
m_new  = evaluate_anomalies(anomalies_new, full_df)

comparison = pd.DataFrame({
    'Baseline (RMSE Strong)':       m_base,
    'With Typology Features':       m_new,
})
comparison['Delta'] = (comparison['With Typology Features'] -
                       comparison['Baseline (RMSE Strong)']).round(3)
print(comparison.to_string())

                              Baseline (RMSE Strong)  With Typology Features   Delta
Anomalies found                              450.000                 478.000  28.000
Unique products                               51.000                  65.000  14.000
Unique days                                   30.000                  30.000   0.000
Anomalies/day (avg)                           15.000                  15.900   0.900
Mean deviation from norm (%)                -116.800                -111.200   5.600
Mean |z_hist|                                  4.146                   4.002  -0.144
Mean |z_july|                                  3.056                   3.053  -0.003
% |z_hist| > 2                                54.400                  55.200   0.800
% |z_hist| > 3                                40.900                  41.200   0.300
% zero sales                                  20.400                  19.700  -0.700
% drops > 50% below norm                      21.100             

## Conclusions

The experiment demonstrates that incorporating demand typology features produces a measurable but mixed effect on anomaly detection quality.

**Coverage improved.** The typology model flagged 478 anomalies across 65 unique products, compared to 450 anomalies across 51 products in the baseline — an increase of 6.2% in alert volume and 27% in product coverage. This suggests that encoding each product's seasonal archetype helps the model better characterise expected demand, allowing it to detect deviations in products that the baseline model could not distinguish from noise.

**Signal strength marginally declined.** Mean |z_hist| decreased from 4.146 to 4.002, and the mean deviation from historical norm narrowed slightly from −116.8% to −111.2%. This indicates that the additional anomalies introduced by the typology model are real but somewhat less statistically extreme than those found by the baseline — consistent with the model detecting a broader set of genuine anomalies rather than introducing noise.

**Core quality constraints held.** Both models maintain 100% stock filter compliance, 100% detection within business hours, and zero out-of-stock false positives. The typology extension does not degrade the operational validity of any detected event.

**Practical recommendation.** The baseline RMSE Strong configuration remains the primary recommended pipeline due to its stronger per-anomaly statistical signal. The typology-augmented model is preferable when maximising product coverage is the operational priority — for example, during catalogue-wide shelf audits where missing a product is costlier than investigating a marginal alert. A natural direction for future work is using the demand typology not as model input features but as a stratification layer for per-archetype threshold calibration, which would allow independent precision-recall tuning for summer products, holiday-driven SKUs, and weekend-peaked items without retraining the underlying model.